In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict
import chromadb

In [ ]:
import os 
from dotenv import load_dotenv,find_dotenv

load_dotenv(find_dotenv(),override=True)

if os.environ["GOOGLE_API_KEY"]:
    print('api key found')
else:
    print("Key not found")


# Create LLM

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.7
)

# Chroma Setup

In [ ]:
client = chromadb.Client()

collection = client.create_collection("jokes")

Define Schema

In [ ]:
class AgentState(TypedDict):
    topic: str
    joke: str

# Joke Generator Function

In [ ]:
def generate_joke(state):

    topic = state["topic"]

    response = llm.invoke(
        [HumanMessage(content=f"Tell me a funny joke about {topic}")]
    )

    joke = response.content

    # Store in Chroma
    collection.add(
        documents=[joke],
        ids=[topic]
    )

    return {
        "topic": topic,
        "joke": joke
    }

# Build LangGraph

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("joke_generator", generate_joke)

builder.set_entry_point("joke_generator")

builder.add_edge("joke_generator", END)

graph = builder.compile()

# Run the Agent

In [ ]:
result = graph.invoke({
    "topic": "programmers"
})

print(result["joke"])

# View Stored Jokes

In [ ]:
data = collection.get()

print(data["documents"])